# Bradford Bulls — Logo Detector (YOLO26m) trên Google Colab

Pipeline:
1. Cài `ultralytics` + `albumentations` + `wandb`.
2. (Tùy chọn) Mount Drive + đăng nhập wandb.
3. **Tải dataset YOLO từ Roboflow** (code của bạn).
4. **Gộp 32→17 brand + re-split clip-aware ~18% val** (Roboflow export là 32 class + split random → phải xử lý lại để metric trung thực và class cân bằng).
5. Train `yolo26m` @ 1280 với augmentation tuned (fliplr=0, hsv_h thấp...) + **MotionBlur**.

> **Runtime → Change runtime type → GPU**. T4 (16GB): `BATCH=8`; A100/L4: `BATCH=16`.

## 0. Cài thư viện + GPU

In [ ]:
!nvidia-smi
!pip -q install -U ultralytics albumentations wandb roboflow
import ultralytics, albumentations, wandb
print('ultralytics', ultralytics.__version__, '| albumentations', albumentations.__version__, '| wandb', wandb.__version__)

In [ ]:
# (Tùy chọn) Mount Drive để checkpoint không mất khi Colab ngắt phiên
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/bradford_logo_runs'
else:
    PROJECT = '/content/runs'
print('Runs ->', PROJECT)

In [ ]:
# Đăng nhập wandb (sẽ hỏi API key — lấy ở https://wandb.ai/authorize)
# Hoặc set thẳng: wandb.login(key='YOUR_WANDB_KEY')
import wandb
wandb.login()
from ultralytics import settings
settings.update({'wandb': True})   # bật tích hợp wandb của ultralytics
print('wandb enabled in ultralytics:', settings.get('wandb'))

## 1. Tải dataset YOLO từ Roboflow (code của bạn)

In [ ]:
# Tải dataset từ Roboflow qua link export trực tiếp
!rm -rf /content/roboflow && mkdir -p /content/roboflow
!curl -L "https://app.roboflow.com/ds/sFwDW9lPqg?key=Q1XfLdv761" -o /content/roboflow.zip
!unzip -q -o /content/roboflow.zip -d /content/roboflow && rm /content/roboflow.zip

EXPORT_DIR = '/content/roboflow'   # chứa data.yaml + train/valid/test
import os
print('Nội dung:', os.listdir(EXPORT_DIR))
assert os.path.exists(os.path.join(EXPORT_DIR, 'data.yaml')), 'Khong thay data.yaml — kiem tra link/format'

## 2. Gộp 32→17 brand + re-split clip-aware

Đọc tên class từ `data.yaml` của bản export, strip `_home/_away` → 17 brand đúng thứ tự pipeline. Gom toàn bộ train+valid+test rồi chia lại theo **cụm trận/clip** (giữ nguyên cả trận về một bên → không rò rỉ frame ~2 fps). Giữ ảnh background.

In [ ]:
import re, shutil, random, yaml
from collections import Counter, defaultdict
from pathlib import Path

OUT_DIR  = '/content/data'
VAL_FRAC = 0.18
DROP_BACKGROUNDS = False
IMG_EXT = {'.jpg', '.jpeg', '.png'}

BRAND_ORDER = ['acs_group','aon','atm','bartercard','cch','chadlaw','ellgren',
    'em_workwear','fairway','floor_tonic','klg','mcp','mna_cladding',
    'mna_support_service','paints_lacquers','romantica','top_notch']
BRAND_IDX = {b: i for i, b in enumerate(BRAND_ORDER)}

def brand_of(name):
    return re.sub(r'_(home|away)$', '', name)

def group_key(stem):
    m = re.match(r'^(M\d+)', stem)
    if m: return m.group(1)
    m = re.match(r'^(clip_\d+)', stem)
    if m: return m.group(1)
    return stem

# 1) map old class index -> new brand index (None = drop, vd placeholder)
cfg = yaml.safe_load((Path(EXPORT_DIR)/'data.yaml').read_text())
old_names = cfg['names']
if isinstance(old_names, dict):                       # ultralytics-style {0: name, ...}
    old_names = [old_names[k] for k in sorted(old_names, key=int)]
old_to_new = {i: BRAND_IDX.get(brand_of(n)) for i, n in enumerate(old_names)}
dropped = [n for i, n in enumerate(old_names) if old_to_new[i] is None]
print(f'{len(old_names)} class export -> {len(BRAND_ORDER)} brand. Dropped: {dropped}')

# 2) gom toàn bộ samples tu moi split
samples = []
for sp in ['train', 'valid', 'val', 'test']:
    idir = Path(EXPORT_DIR)/sp/'images'; ldir = Path(EXPORT_DIR)/sp/'labels'
    if not idir.exists(): continue
    for img in sorted(idir.glob('*')):
        if img.suffix.lower() in IMG_EXT:
            samples.append((img, ldir/(img.stem + '.txt')))
print('Tong so anh gom duoc:', len(samples))

def read_new_classes(lbl):
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if line:
                new = old_to_new.get(int(line.split()[0]))
                if new is not None: out.append(new)
    return out

# 3) gom theo cum (group) + dem class
groups = defaultdict(lambda: {'items': [], 'counts': Counter()})
for img, lbl in samples:
    g = groups[group_key(img.stem)]
    g['items'].append((img, lbl))
    for c in read_new_classes(lbl): g['counts'][c] += 1
gkeys = sorted(groups); total = len(samples)
totals = Counter()
for g in groups.values(): totals.update(g['counts'])

# 4) tim split clip-aware ~VAL_FRAC, uu tien giu class hiem o train
def one_split(seed):
    order = list(gkeys); random.Random(seed).shuffle(order)
    val, n = set(), 0
    for k in order:
        if n >= VAL_FRAC*total: break
        val.add(k); n += len(groups[k]['items'])
    return val
def score(valset):
    tr, va = Counter(), Counter()
    for k, g in groups.items():
        (va if k in valset else tr).update(g['counts'])
    bad = sum(1 for c in totals if tr[c] < 0.6*totals[c])
    nval = sum(len(groups[k]['items']) for k in valset)
    return (bad, abs(nval/(total or 1) - VAL_FRAC))
best = min((one_split(s) for s in range(300)), key=score)

# 5) ghi ra OUT_DIR (remap class id, giu background = file rong)
out = Path(OUT_DIR)
if out.exists(): shutil.rmtree(out)
for sp in ('train', 'val'):
    (out/sp/'images').mkdir(parents=True, exist_ok=True)
    (out/sp/'labels').mkdir(parents=True, exist_ok=True)
counts = Counter(); cls_per = {'train': Counter(), 'val': Counter()}
for k, g in groups.items():
    sp = 'val' if k in best else 'train'
    for img, lbl in g['items']:
        is_bg = not read_new_classes(lbl)
        if is_bg and DROP_BACKGROUNDS: continue
        shutil.copy2(img, out/sp/'images'/img.name)
        lines = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                line = line.strip()
                if not line: continue
                parts = line.split(); new = old_to_new.get(int(parts[0]))
                if new is None: continue
                parts[0] = str(new); lines.append(' '.join(parts)); cls_per[sp][new] += 1
        (out/sp/'labels'/(img.stem + '.txt')).write_text('\n'.join(lines))
        counts[sp] += 1

names_str = '[' + ', '.join(f"'{n}'" for n in BRAND_ORDER) + ']'
(out/'data.yaml').write_text(
    f'path: {out.as_posix()}\ntrain: train/images\nval: val/images\n'
    f'nc: {len(BRAND_ORDER)}\nnames: {names_str}\n')

print(f'Split: {counts["train"]} train / {counts["val"]} val ({counts["val"]/(sum(counts.values()) or 1):.0%} val)')
print('Val groups:', sorted(best))
print('\nPer-brand (train | val):')
for i, b in enumerate(BRAND_ORDER):
    print(f'  {i:2d} {b:24s} {cls_per["train"][i]:5d} | {cls_per["val"][i]:4d}')
print('\ndata.yaml ->', out/'data.yaml')

## 3. Train YOLO26m (tuned aug + MotionBlur + wandb)

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('WANDB_PROJECT', 'bradford-logo-detection')

# ---- cau hinh (A100/H100) ----
MODEL    = 'yolo26m.pt'   # 'yolo26l.pt' neu muon thu capacity lon hon
IMGSZ    = 1536           # logo nho/tiny -> resolution cao la don bay mAP lon nhat (1280 -> 1536)
EPOCHS   = 300
PATIENCE = 60
BATCH    = 16             # A100-40GB @1536: ~16 | A100-80GB/H100: 24-32. Tang neu con VRAM
CACHE    = 'ram'          # Colab high-RAM (A100/H100) du RAM -> nhanh nhat
MB_P     = 0.30           # xac suat MotionBlur (0 = tat)
NAME     = 'logo_yolo26m_colab'

def patch_motion_blur(mb_p=0.30, blur_limit=(3, 15)):
    """Chen MotionBlur (pixel-level, khong doi bbox) vao pipeline Albumentations cua Ultralytics."""
    if mb_p <= 0:
        print('[aug] motion blur OFF'); return
    import albumentations as A
    from ultralytics.data import augment as aug
    _orig = aug.Albumentations.__init__
    def __init__(self, p=1.0, transforms=None):
        if transforms is None:
            transforms = [A.Blur(p=0.01), A.MedianBlur(p=0.01), A.ToGray(p=0.01),
                          A.CLAHE(p=0.01), A.MotionBlur(blur_limit=blur_limit, p=mb_p)]
        _orig(self, p, transforms)
    aug.Albumentations.__init__ = __init__
    print(f'[aug] MotionBlur injected (blur_limit={blur_limit}, p={mb_p})')

# Pre-init wandb voi ten project sach (ultralytics se tai su dung run nay).
import wandb
if wandb.run is None:
    wandb.init(project=os.environ['WANDB_PROJECT'], name=NAME)

from ultralytics import YOLO
patch_motion_blur(MB_P)
model = YOLO(MODEL)
model.train(
    data=f'{OUT_DIR}/data.yaml', imgsz=IMGSZ, pretrained=True,
    project=PROJECT, name=NAME, device=0, seed=0, deterministic=True,
    epochs=EPOCHS, patience=PATIENCE, batch=BATCH, optimizer='auto', cos_lr=True,
    lr0=0.01, lrf=0.01, weight_decay=0.0005, warmup_epochs=3.0,
    close_mosaic=25, cache=CACHE,
    box=7.5, cls=0.5, dfl=1.5,
    hsv_h=0.010, hsv_s=0.70, hsv_v=0.50, degrees=5.0, translate=0.10, scale=0.50,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0, mosaic=1.0, mixup=0.10,
    copy_paste=0.0, erasing=0.40, val=True, plots=True,
)

## 4. Đánh giá honest + tải `best.pt`

In [ ]:
from ultralytics import YOLO
best = f'{PROJECT}/{NAME}/weights/best.pt'
m = YOLO(best).val(data=f'{OUT_DIR}/data.yaml', imgsz=IMGSZ, plots=True)
print(f'mAP50-95 {m.box.map:.4f}   mAP50 {m.box.map50:.4f}   P {m.box.mp:.3f}  R {m.box.mr:.3f}')
print('Per-class mAP50-95:')
for i, ap in enumerate(m.box.maps):
    print(f'  {m.names[i]:<24} {ap:.4f}')
print('\nbest.pt ->', best)

In [ ]:
# Tai best.pt ve may (neu khong mount Drive). Neu da mount Drive thi no da nam san trong Drive.
from google.colab import files
files.download(best)